In [9]:
#Array packages
import pandas as pd
import numpy as np
import xarray as xr
import netCDF4 as nc4

from scipy.stats import kendalltau
import pymannkendall as mk

#plots
import matplotlib.pyplot as plt
import rioxarray as rio
import geopandas as gpd
from shapely.geometry import mapping
import cartopy.crs as ccrs

#Progress meter
from dask.diagnostics import ProgressBar
from tqdm import tqdm

# Directories
import os
import glob
import dask
#import h5netcdf
import scipy

import scipy.stats as stats
import matplotlib.pyplot as plt
import itertools
import pandas as pd

import os
os.chdir(r"G:\OneDrive - IIT Delhi\3. IIT DELHI\2. Research\2_Papers\1_Clustering connectivity")
#os.chdir(r"E:\academy\OneDrive - IIT Delhi\3. IIT DELHI\2. Research\2_Papers\1_Clustering connectivity")

print(os.getcwd())

G:\OneDrive - IIT Delhi\3. IIT DELHI\2. Research\2_Papers\1_Clustering connectivity


### 1. PPT_SF connectedness

In [6]:
gauge_info=pd.read_csv(r"3_Data/Data_p/2_Station/1_Streamflow_data/gauge_info_p.csv")
gauge_info['Period_length']=gauge_info['Expected_entries']/365

# Final station criteria
yr_strt=1980;yr_end=2020
period=22; miss_prc=4
gauge_final=gauge_info.loc[(gauge_info['Period_length']>period) & (gauge_info['missing_percent']<miss_prc)]
gauge_final.set_index('GaugeID',inplace=True)



# Threshold criteria
th=98.5
ICT=10 #days
tau=7

# Variable initialization
ssn=['All','Monsoon','Post-monsoon']
ssn_months=[[1,2,3,4,5,6,7,8,9,10,11,12],[6,7,8,9],[10,11,12]]

dates=pd.date_range(start='1950',end='2022',freq='D')
strm_events_matrix=pd.DataFrame(np.nan,index=dates,columns=gauge_final.index.values)
ppt_events_matrix=pd.DataFrame(np.nan,index=dates,columns=gauge_final.index.values)

# Loading data
ds=xr.open_dataset("3_Data/Data_p/4_PPT IMD/PPT_station.nc")

null_stations=[]

for s,seas in enumerate(ssn_months[0:1]):

    for stn in tqdm(gauge_final.index.values):

        df=ds.sel(Station=gauge_final.loc[stn,'Station']).to_dataframe()
        strm_flw=pd.read_csv(f'3_Data/Data_p/2_Station/1_Streamflow_data/{stn}.csv',index_col=0,parse_dates=True)
        time_clip=pd.date_range(start=strm_flw.index.min(),end=strm_flw.index.max(),freq='D')

        data=df.loc[time_clip,:]
        data_ss=data.loc[data.index.month.isin(seas),:]

        
        if (data_ss[['Ppt','Streamflow']].isna().all().sum() == 0):          # At gauge 10, ppt have full nan value
            
            strm_events = event_identification(data_ss.copy(),th,ICT,'Streamflow')      
            ppt_events  = event_identification(data_ss.copy(),th,ICT,'Ppt')
            
            strm_events_matrix.loc[strm_events['Peak Date'],stn]=1
            ppt_events_matrix.loc[ppt_events['Peak Date'],stn]=1

        else:
            null_stations.append(stn)
            strm_events_matrix.drop(columns=stn,inplace=True)
            ppt_events_matrix.drop(columns=stn,inplace=True)
            gauge_final.drop(stn,inplace=True)

    strm_events_matrix.to_csv(fr"2_Analysis\3_Spatial connectivity\2_ppt_sf_connectedness\Outputs\strm_events_{ssn[s]}.csv")
    ppt_events_matrix.to_csv(fr"2_Analysis\3_Spatial connectivity\2_ppt_sf_connectedness\Outputs\ppt_events_{ssn[s]}.csv")


    #connect=connectedness(strm_events_matrix,ppt_events_matrix,tau,gauge_final)
    #ppt_connect.to_csv(fr"2_Analysis\3_Spatial connectivity\2_ppt_sf_connectedness\Outputs\ppt_ES_{ssn[s]}.csv")
    #gauge_final.to_csv(r"2_Analysis\3_Spatial connectivity\2_ppt_sf_connectedness\Outputs\gauge_info.csv")

  5%|▍         | 10/211 [00:01<00:32,  6.22it/s]C:\Users\2024CEZ8029\AppData\Local\Temp\ipykernel_14468\2669580980.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gauge_final.drop(stn,inplace=True)
100%|██████████| 211/211 [00:31<00:00,  6.63it/s]


## Functions

### 1 Event identification

In [3]:
def event_identification (data1,th,ICT,var):


    th_value=np.nanpercentile(data1[var].dropna(),th)

    data1['exceedence']=(data1[var]>th_value).astype(int)
    data1['change point']=data1['exceedence'].diff().fillna(0)


    data1['flood start']=data1['change point']==1
    data1['event']=data1['flood start'].cumsum()

    floods=data1[data1['exceedence']==1]
    flood_events=floods.groupby('event')


    results = []
    for event, group in flood_events:
        start = group.index[0]
        end = group.index[-1]
        duration = (end - start).days+1
        peak = group[var].max()
        peak_date = group[var].idxmax() 
        severity = group[var].sum()
        
        results.append([start, end, duration,severity, peak,peak_date])


    flood_summary = pd.DataFrame(results, columns=['Start', 'End', 'Duration (days)','Severity', 'Peak Flow','Peak Date'])


    # Handling dependent events (Event occuring in a few days gap)
    flood_summary["Gap (days)"] = flood_summary["Start"]-flood_summary["End"].shift(1) #Calculate gap between consecutive events
    flood_summary["Gap (days)"] = flood_summary["Gap (days)"].dt.days  # Convert to integer days
    flood_summary['Gap binary'] = (flood_summary["Gap (days)"] < ICT).astype(int)
    flood_summary["Group"] = (flood_summary["Gap binary"] == 0).cumsum()
    merged_floods = flood_summary.groupby("Group").agg({
        "Start": "min",  # Earliest start date in the group
        "End": "max",  # Latest end date in the group
        "Duration (days)": "sum",  # Sum durations
        "Severity": "sum",  # Sum severities
        "Peak Flow": "max",  # Max peak flow in the group
        "Peak Date": "min"  # Earliest peak date in the group
    }).reset_index()

    merged_floods=merged_floods.iloc[0:-1,0:7]
    
    return merged_floods


### 2 ppt-streamflow connectedness

In [ ]:
strm=strm_events_matrix
ppt=ppt_events_matrix


#def connectedness2(strm,ppt,tau,gauge_final):
    
stations = gauge_final.index.values

tau=6

for stn in tqdm(stations):

    #Clip the common dates
    clip_st=gauge_final.loc[stn,'Start_date']
    clip_end=gauge_final.loc[stn,'End_date']

    strm_clip=strm.loc[clip_st:clip_end]
    ppt_clip=ppt.loc[clip_st:clip_end]

    # 1. strm (y)/ppt(x) (X is in flood and what is the chance that Y being in flood)
    event_x=ppt_clip[ppt_clip[stn]==1].copy()
    sum_x=event_x[stn].sum()

    date_window = sorted(set([d + pd.Timedelta(days=i) for d in event_x.index for i in range(0, tau)])) # Set helps to store only unique date
    datetime_index = pd.DatetimeIndex(date_window)
    yx_forward=strm.loc[datetime_index,:][stn].sum()

    # 3 Total synchornization
    gauge_final.loc[stn,'ES']=(yx_forward)/(sum_x)


    delay=[]
    for x_date in event_x.index:
        
        date_window = sorted(set([x_date + pd.Timedelta(days=i) for i in range(1 - tau, tau)]))
        
        events=strm.loc[pd.DatetimeIndex(date_window),:]
        y_date=events[events[stn]==1].index
        
        if pd.notna(y_date.mean()):
            delay.append((y_date.mean() - x_date).days)  # sometime more than one event of y will come in the window, so take mean

    # Compute the average delay
    gauge_final.loc[stn, "Delay"] = np.mean(delay)
                                                                                                                                                                                       


100%|██████████| 210/210 [00:20<00:00, 10.00it/s]


In [60]:
event_x

,IWM-gauge-17,IWM-gauge-27,IWM-gauge-29,IWM-gauge-32,IWM-gauge-36,IWM-gauge-40,IWM-gauge-54,IWM-gauge-57,IWM-gauge-60,IWM-gauge-68,...,IWM-gauge-995,IWM-gauge-996,IWM-gauge-997,IWM-gauge-999,IWM-gauge-1000,IWM-gauge-1001,IWM-gauge-1004,IWM-gauge-1006,IWM-gauge-1078,IWM-gauge-1103
1990-05-10,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1990-06-16,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1990-06-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1990-08-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1990-08-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-06-26,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2019-07-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2020-06-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
2020-07-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
